In [ ]:
%cd ..

# Gramians Analysis

In [ ]:
from dataclasses import dataclass
import pickle
import numba as nb
import numpy as np
import scipy.stats as stats
import mdtraj as md

@dataclass(slots=True, frozen=True)
class NBConfig:
    # Gramian rank-1 decomposition settings
    n_components: int = 2
    subsample: int = 1

In [ ]:
def compute_rmsd(ca_trajectory: md.Trajectory) -> np.ndarray:
    RMSD = []
    for i in range(len(ca_trajectory)):
        RMSD.append(md.rmsd(ca_trajectory, ca_trajectory, i)) # type: ignore
    RMSD = np.ascontiguousarray(RMSD)

    return RMSD

@nb.njit(nogil=True)
def _lddt(x: np.ndarray, y:np.ndarray, threshold: float=15.):
    n_dim = x.shape[-2]

    score = 0.0
    contacts = 0
    for i in range(n_dim):
        for j in range(i+1, n_dim):
            dx = np.sqrt(((x[i]-x[j]) ** 2).sum())
            dy = np.sqrt(((y[i]-y[j]) ** 2).sum())
            l1 = np.abs(dx - dy)

            if dx < threshold:
                score += 0.25 * (int(l1 < 0.5) + int(l1 < 1.0) + int(l1 < 2.0) + int(l1 < 4.0)) * 2
                contacts += 1 * 2

    return score / contacts

@nb.njit(nogil=True, parallel=True)
def lddt(x: np.ndarray, y:np.ndarray, threshold: float=15.):
    n_dim = x.shape[-2]

    if x.ndim == 2:
        x_batch_shape = np.array([1], dtype=np.int64)
    else:
        x_batch_shape = np.array(x.shape[:-2], dtype=np.int64)

    if y.ndim == 2:
        y_batch_shape = np.array([1], dtype=np.int64)
    else:
        y_batch_shape = np.array(y.shape[:-2], dtype=np.int64)

    x_batch_size = 1
    for s in x_batch_shape:
        x_batch_size *= s

    y_batch_size = 1
    for s in y_batch_shape:
        y_batch_size *= s

    flat_x = x.reshape(x_batch_size, n_dim, 3)
    flat_y = y.reshape(y_batch_size, n_dim, 3)
    flat_res = np.empty((x_batch_size * y_batch_size,), dtype=x.dtype)

    for i in nb.prange(x_batch_size * y_batch_size):
        j = i // y_batch_size
        k = i % y_batch_size

        flat_res[i] = _lddt(flat_x[j], flat_y[k], threshold)

    return flat_res.reshape(x.shape[:-2] + y.shape[:-2])

def compute_lddt(ca_trajectory) -> np.ndarray:
    ca_trajectory_xyz = np.ascontiguousarray(ca_trajectory.xyz)

    lDDT = lddt(ca_trajectory_xyz, ca_trajectory_xyz)

    return lDDT

In [ ]:
def _compute_gramian(data: np.ndarray, config: NBConfig) -> np.ndarray:
    """Compute Gramian analysis on the given data."""
    flat_data_subsampled = np.reshape(data, (data.shape[0], -1))[::config.subsample]

    gramian = np.einsum(
        "ik,jk->ij", flat_data_subsampled, flat_data_subsampled
    )

    return gramian

def _decompose_gramian(gramian: np.ndarray, config: NBConfig) -> np.ndarray:
    L, Q = np.linalg.eigh(gramian)
    Q = Q * np.sqrt(np.maximum(L, 0)[None])
    L, Q = L[::-1], Q[:, ::-1]

    components = []
    for i in range(config.n_components):
        component = Q[:, None, i] @ Q[:, None, i].T
        max_score = np.max(np.abs(component))
        component_normalized = component / max_score
        components.append(component_normalized)

    components = np.array(components)

    return components

def compute_gramian(features: dict[str, np.ndarray], config: NBConfig):
    """Compute Gramians"""
    result = {}
    for representation, data in features.items():
        gramian = _compute_gramian(data, config)
        gramian_components = _decompose_gramian(gramian, config)

        result[representation] = {
            "gramian": gramian,
            "components": gramian_components
        }

    return result

In [ ]:
def _compute_correlation(matrixA: np.ndarray, matrixB: np.ndarray):
    if matrixB is None:
        return np.nan

    if matrixA.shape != matrixB.shape:
        raise ValueError(f"matrix A shape ({matrixA.shape}) should be the same as matrix B ({matrixB.shape})")


    tril_indices = np.tril_indices_from(matrixA, k=-1)

    matrixA_flat = matrixA[tril_indices]
    matrixB_flat = matrixB[tril_indices]

    correlation_coef = stats.spearmanr(matrixA_flat, matrixB_flat).statistic # type: ignore

    return correlation_coef

def compute_stats(gramian: dict[str, np.ndarray], rmsd_matrix: np.ndarray, lddt_matrix: np.ndarray):
    rmsd_corr_coef = _compute_correlation(gramian["gramian"], rmsd_matrix)
    lddt_corr_coef = _compute_correlation(gramian["gramian"], lddt_matrix)

    components_rmsd_corr_coef = []
    components_lddt_corr_coef = []

    for component in gramian["components"]:
        components_rmsd_corr_coef.append(
            _compute_correlation(component, rmsd_matrix),
        )
        components_lddt_corr_coef.append(
            _compute_correlation(component, lddt_matrix)
        )

    components_rmsd_corr_coef = np.array(components_rmsd_corr_coef)
    components_lddt_corr_coef = np.array(components_lddt_corr_coef)

    gramian_stats = {
        "RMSDCorrCoef": rmsd_corr_coef,
        "lDDTCorrCoef": lddt_corr_coef,
        "ComponentsRMSDCorrCoef": components_rmsd_corr_coef,
        "ComponentslDDTCorrCoef": components_lddt_corr_coef
    }

    return gramian_stats

In [ ]:
# Warmup numba functions
def _warmup_numba():
    """Compile numba functions."""
    X = np.random.randn(10, 10, 3)
    Y = np.random.randn(10, 10, 3)
    lddt(X, Y)

_warmup_numba()

In [ ]:
top_pdb = "examples/_structure.pdb"
traj_dcd = "examples/_trajectory.dcd"

config = NBConfig()

In [ ]:
# Load trajectory
traj = md.load(traj_dcd, top=top_pdb)[::config.subsample]
ca_traj = traj.atom_slice(traj.top.select("protein and name CA"))

# Load reference as trajectory
ref = md.load(top_pdb)
ca_ref = ref.atom_slice(ref.top.select("protein and name CA"))

# Center & align
ca_ref.center_coordinates()
ca_traj.center_coordinates()
ca_traj.superpose(ca_ref, 0)

with open("examples/features.pkl", "rb") as h:
    features = pickle.load(h)["data"]

In [ ]:
# Reference measures
ca_rmsd = compute_rmsd(ca_traj) * 10
ca_lddt = compute_lddt(ca_traj)

# Gramians
gramians = compute_gramian(features, config)

# Stats
stats = {
    f: compute_stats(g, ca_rmsd, ca_lddt) for f, g in gramians.items()
}

with open("examples/gramians.pkl", "wb") as h:
    pickle.dump({
        "data": features,
        "stats": stats
    }, h, protocol=pickle.HIGHEST_PROTOCOL)